In [1]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

2026-03-03 17:45:00.558261: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-03 17:45:01.013405: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-03 17:45:02.952343: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install seaborn
!pip install scipy
!pip install scikit-learn
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install tensorflow
!pip install keras --upgrade


import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu126
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


# ------------------------------------- FINAL PIPELINE ---------------------------------

### ------- Merging the CSVs at data/MachineLearningCVE/*.csv --------

In [3]:
!pip install glob2
import glob

Defaulting to user installation because normal site-packages is not writeable


In [4]:
all_files = glob.glob('../data/MachineLearningCVE/*.csv')
df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)

In [5]:
print(df[' Label'].value_counts())

 Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [6]:
#df.to_csv('../data/full_ds.csv', index=False)

### -------------------------------- Preprocessing ------------------------------------

In [7]:
from sklearn.preprocessing import LabelEncoder, RobustScaler
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
import joblib

In [8]:
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[' Label'])

In [9]:
label_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
label_mapping

{'BENIGN': np.int64(0),
 'Bot': np.int64(1),
 'DDoS': np.int64(2),
 'DoS GoldenEye': np.int64(3),
 'DoS Hulk': np.int64(4),
 'DoS Slowhttptest': np.int64(5),
 'DoS slowloris': np.int64(6),
 'FTP-Patator': np.int64(7),
 'Heartbleed': np.int64(8),
 'Infiltration': np.int64(9),
 'PortScan': np.int64(10),
 'SSH-Patator': np.int64(11),
 'Web Attack � Brute Force': np.int64(12),
 'Web Attack � Sql Injection': np.int64(13),
 'Web Attack � XSS': np.int64(14)}

In [10]:
y = to_categorical(df['label_encoded'])

In [11]:
X = df.drop(columns=[' Label', 'label_encoded']).apply(pd.to_numeric, errors='coerce')
X.replace([np.inf, -np.inf], np.nan, inplace=True)
valid_indices = X.dropna().index

X = X.loc[valid_indices]
y = y[valid_indices]
label_encoded = df['label_encoded'].loc[valid_indices]

In [12]:
rs = RobustScaler()
X_scaled = rs.fit_transform(X)

## --------------------------------- Creation du modèle. -------------------------------------------

In [13]:
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

X_train.shape

y_train.shape

num_classes = y_train.shape[1]

"""
banana_brain_0 = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dropout(0.1),
    
    Dense(num_classes, activation='softmax')
])
"""

opt = Adam(learning_rate=0.00001) 

banana_brain_0.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

In [14]:
from sklearn.utils.class_weight import compute_class_weight


history = banana_brain_0.fit(
    X_train, y_train,
    epochs=50,
    batch_size=512,
    validation_split=0.15,
    callbacks=[early_stop, lr_scheduler, checkpoint],
    class_weight=class_weight_dict
)



X_train_small = X_train
y_train_small = y_train

train_dataset = tf.data.Dataset.from_tensor_slices((X_train_small, y_train_small))
train_dataset = train_dataset.shuffle(10000).batch(512).prefetch(tf.data.AUTOTUNE)

history = banana_brain_0.fit(
    train_dataset,
    epochs=50,
    validation_data=(X_test, y_test),
    callbacks=[early_stop, lr_scheduler, checkpoint]
)


test_loss, test_accuracy = banana_brain_0.evaluate(X_test, y_test)
print(f"Test accuracy: {test_accuracy:.4f}")

from sklearn.metrics import classification_report

y_pred = banana_brain_0.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes, target_names=le.classes_))

# ----------------------------- Breach for 90% ----------------------------------

In [15]:
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

In [16]:
# Start fresh from scaled data
# Only apply SMOTE to minority classes that need it
# First, separate majority (BENIGN) from minorities

benign_mask = label_encoded == 0  # BENIGN is class 0
X_benign = X_scaled[benign_mask]
y_benign = label_encoded[benign_mask]

X_attacks = X_scaled[~benign_mask]
y_attacks = label_encoded[~benign_mask]

# Small ass memory thus downsampling to 270k samples...
np.random.seed(42)
benign_indices = np.random.choice(len(X_benign), size=270000, replace=False)
X_benign_sampled = X_benign[benign_indices]
y_benign_sampled = y_benign.iloc[benign_indices]

# Merging back
X_reduced = np.vstack([X_benign_sampled, X_attacks])
y_reduced = np.concatenate([y_benign_sampled, y_attacks])

# SMOTE only on attacks (smaller dataset)
benign_mask_reduced = y_reduced == 0
X_benign_final = X_reduced[benign_mask_reduced]
y_benign_final = y_reduced[benign_mask_reduced]
X_attacks_reduced = X_reduced[~benign_mask_reduced]
y_attacks_reduced = y_reduced[~benign_mask_reduced]

smote = SMOTE(random_state=42, k_neighbors=5, sampling_strategy='auto')

X_attacks_smoted, y_attacks_smoted = smote.fit_resample(X_attacks_reduced, y_attacks_reduced)

# Combine back with BENIGN
X_final = np.vstack([X_benign_final, X_attacks_smoted])
y_final = np.concatenate([y_benign_final, y_attacks_smoted])

print(f"Original: {X_benign.shape[0]} rows")
print(f"After SMOTE: {X_final.shape[0]} rows")
print(np.bincount(y_final))

Original: 2271320 rows
After SMOTE: 3491736 rows
[270000 230124 230124 230124 230124 230124 230124 230124 230124 230124
 230124 230124 230124 230124 230124]


In [17]:
print("y",type(y_benign), "X",type(X_benign))

y <class 'pandas.core.series.Series'> X <class 'numpy.ndarray'>


In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final,
    test_size=0.2, 
    random_state=42, 
    stratify=y_final
)
print(f"Avant one hot -> to_categorical")
print(f"y_train[:5]: {y_train[:5]}")
print(f"y_train type: {type(y_train)}")
print(f"y_train shape: {y_train.shape}")

y_train_onehot = to_categorical(y_train)
y_test_onehot = to_categorical(y_test)

print(f"Après one hot -> to_categorical")
print(f"y_onehot_train type: {type(y_train_onehot)}")
print(f"y_onehot_train shape: {y_train_onehot.shape}")

Avant one hot -> to_categorical
y_train[:5]: [ 1 12  0  3  4]
y_train type: <class 'numpy.ndarray'>
y_train shape: (2793388,)
Après one hot -> to_categorical
y_onehot_train type: <class 'numpy.ndarray'>
y_onehot_train shape: (2793388, 15)


In [ ]:
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"y_onehot: {y_train_onehot.shape}, Test: {y_test_onehot.shape}")

In [ ]:
opt = Adam(learning_rate=0.0001)
def focal_loss(gamma=2.0, alpha=0.25):
    def loss_func(y_true, y_pred):
        epsilon = tf.keras.backend.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * tf.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return tf.reduce_sum(loss, axis=-1)
    return loss_func

In [ ]:
num_classes = y_train_onehot.shape[1]

banana_brain_0_nuked = Sequential([ 
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)), 
    BatchNormalization(), 
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),
    
    Dense(32, activation='relu'),
    Dropout(0.1),
    
    Dense(num_classes, activation='softmax')
])

In [ ]:
banana_brain_0_nuked.compile(
    optimizer=opt, 
    loss=focal_loss(gamma=2.0, alpha=0.25), 
    metrics=['accuracy']
)

In [ ]:
print(f"X_train Size: {X_train.nbytes / 1024**3:.2f} GB")
print(f"y_train_onehot.nbytes Size: {y_train_onehot.nbytes / 1024**3:.2f} GB")
print(f"X_train.shape: {X_train.shape}")

In [ ]:
class_weights = compute_class_weight(
    'balanced', 
    classes=np.unique(y_train), 
    y=y_train)

class_weight_dict = {}
for i, weight in enumerate(class_weights):
    if weight > 5:
        class_weight_dict[i] = weight * 5
    else:
        class_weight_dict[i] = weight
print(f"Class Weights: {class_weight_dict}")

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5,
    patience=3, 
    min_lr=1e-7)

checkpoint = ModelCheckpoint(
    'best_model_nuked.keras', 
    monitor='val_loss', 
    save_best_only=True)

In [ ]:
print(num_classes)
print(len(np.unique(y_train)))
print(list(class_weight_dict.keys()))

In [ ]:
print(f"\n======= CREATION DATASET ============")
print(f"X_train type: {type(X_train)}")
print(f"y_train_onehot type: {type(y_train_onehot)}")

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_onehot))
print(f"Dataset créé: {train_dataset}")

train_dataset = train_dataset.shuffle(500).batch(32).prefetch(tf.data.AUTOTUNE)
print(f"Dataset après shuffle/batch/prefetch: {train_dataset}")

tf.debugging.set_log_device_placement(True)
print(f"\n======== TEST BATCH N°1 ==========")
for batch_x, batch_y in train_dataset.take(1):
    print(f"Batch x shape: {batch_x.shape}")
    print(f"Batch y shape: {batch_y.shape}")
    print(f"Batch x type: {batch_x.dtype}")
    print(f"Batch y type: {batch_y.dtype}")
    break
print(f"\n======= FIT ========")
history = banana_brain_0_nuked.fit(
    train_dataset,
    epochs=50,
    validation_data=(X_test, y_test_onehot),
    callbacks=[early_stop, lr_scheduler, checkpoint],
    class_weight=class_weight_dict
)   